# XGBoost height prediction using all visible vertebrae

This notebook predicts one value: masked mean frontal vertebral height. Every other visible vertebra may contribute normalized morphology features.

**Research use only.** This predicts observed height and is not clinically validated.

## Flow

Ordered spine chain → mask target at relative offset 0 → place every remaining vertebra at offset −23…−1 or +1…+23 → leave unavailable offsets as NaN → XGBoost predicts a correction to nearest-neighbor interpolation → evaluate normalized height.

The relative offset is not an anatomical level. It only tells the model how far above or below the target each visible vertebra lies.

In [ ]:
# If needed, uncomment and run once:
# %pip install -r ../../requirement.txt

## 1. Configuration

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from xgboost import XGBRegressor


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "dataset").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.train_all_spine_height_xgboost import (
    ALL_SPINE_FEATURES,
    all_spine_feature_columns,
    build_all_spine_features,
    load_vertebrae,
)
from src.train_masked_height_xgboost import height_metrics, height_targets
from src.train_masked_morphology_xgboost import load_dataset

DATASET_ROOT = REPO_ROOT / "dataset/processed/masked_morphology_coco_nih_lumos"
OUTPUT_DIR = REPO_ROOT / "outputs/masked_morphology/xgboost_all_vertebrae_height_notebook_v1"
MAX_OFFSET = 23
LIMIT_PER_SPLIT = None  # Use 200 for a quick trial.
SAVE_ARTIFACTS = True
SEED = 20260814

print("Dataset:", DATASET_ROOT)
print("Output:", OUTPUT_DIR)

## 2. Load patient/case-separated splits and complete chains

In [ ]:
frames, schema, four_context_columns = load_dataset(
    DATASET_ROOT, limit_per_split=LIMIT_PER_SPLIT
)
chains = {
    split: load_vertebrae(DATASET_ROOT, split)
    for split in ("train", "val", "test")
}
summary = pd.DataFrame({
    split: {
        "samples": len(frame),
        "groups": frame["group_id"].nunique(),
        "images": frame["image_key"].nunique(),
        "maximum_chain_length": max(len(chain) for chain in chains[split].values()),
    }
    for split, frame in frames.items()
}).T
display(summary)

## 3. Build the all-vertebra sparse feature table

Each non-target offset has seven features: left/right height, superior/inferior width, relative orientation, and relative center x/y. Heights and widths use the same context-specific reference scale as training. NaN means that the image has no vertebra at that offset; XGBoost handles NaN natively.

In [ ]:
X = {
    split: build_all_spine_features(
        frame, chains[split], max_offset=MAX_OFFSET
    )
    for split, frame in frames.items()
}
targets = {split: height_targets(frame) for split, frame in frames.items()}

print(f"Feature columns: {X['train'].shape[1]}")
print("Measurements per visible offset:", ALL_SPINE_FEATURES)
print("Target offset 0 present:", any(name.startswith("x_+0_") for name in X["train"].columns))

In [ ]:
example_index = min(100, len(X["train"]) - 1)
example_features = X["train"].iloc[example_index].dropna()
display(example_features.rename("value").to_frame().round(4))

offset_observation = []
for offset in range(-MAX_OFFSET, MAX_OFFSET + 1):
    if offset == 0:
        continue
    offset_columns = [
        name for name in X["train"].columns if name.startswith(f"x_{offset:+d}_")
    ]
    offset_observation.append({
        "relative_offset": offset,
        "observed_train_percent": 100 * X["train"][offset_columns].notna().all(axis=1).mean(),
    })
offset_observation = pd.DataFrame(offset_observation)
display(offset_observation.loc[offset_observation["observed_train_percent"].gt(0)].round(2))

In [ ]:
assert all(name.startswith("x_") for name in X["train"].columns)
assert not any(name.startswith("x_+0_") for name in X["train"].columns)
assert all(np.isfinite(table.dropna().to_numpy()).all() for table in X.values())
assert all(np.isfinite(table.to_numpy()).all() for table in targets.values())

group_sets = {split: set(frame["group_id"]) for split, frame in frames.items()}
assert group_sets["train"].isdisjoint(group_sets["val"])
assert group_sets["train"].isdisjoint(group_sets["test"])
assert group_sets["val"].isdisjoint(group_sets["test"])
print("Audit passed: target excluded, finite observed values, and no group leakage.")

## 4. Train one height-only XGBoost model

The output remains a single residual correction to nearest-neighbor interpolation.

In [ ]:
MODEL_PARAMS = {
    "objective": "reg:pseudohubererror",
    "eval_metric": "mae",
    "n_estimators": 1800,
    "learning_rate": 0.03,
    "max_depth": 3,
    "min_child_weight": 4.0,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.0,
    "reg_lambda": 2.0,
    "tree_method": "hist",
    "device": "cpu",
    "early_stopping_rounds": 80,
    "random_state": SEED,
    "n_jobs": 4,
}
display(pd.Series(MODEL_PARAMS, name="value").to_frame())

In [ ]:
model = XGBRegressor(**MODEL_PARAMS)
model.fit(
    X["train"],
    targets["train"]["mean_height_residual"],
    sample_weight=frames["train"]["sample_weight"].to_numpy(),
    eval_set=[(X["val"], targets["val"]["mean_height_residual"])],
    sample_weight_eval_set=[frames["val"]["sample_weight"].to_numpy()],
    verbose=False,
)
history = model.evals_result()
print("Best iteration:", model.best_iteration)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history["validation_0"]["mae"], color="#219ebc")
ax.axvline(model.best_iteration, color="#fb8500", linestyle="--")
ax.set(xlabel="Boosting iteration", ylabel="Validation residual MAE")
plt.show()

## 5. Evaluate against interpolation and four-context XGBoost

In [ ]:
metric_rows = []
prediction_tables = []
for split in ("val", "test"):
    predicted_residual = model.predict(X[split])
    predicted_height = (
        targets[split]["baseline_mean_height_norm"].to_numpy() + predicted_residual
    )
    metric_rows.append({
        "split": split,
        "method": "xgboost_all_visible_vertebrae",
        **height_metrics(
            targets[split]["actual_mean_height_norm"].to_numpy(),
            targets[split]["baseline_mean_height_norm"].to_numpy(),
            predicted_height,
            frames[split]["sample_weight"].to_numpy(),
        ),
    })
    table = frames[split][[
        "sample_id", "group_id", "image_path", "source_dataset",
        "target_annotation_id", "target_chain_rank", "chain_count", "sample_weight",
    ]].copy()
    table.insert(1, "split", split)
    table["actual_mean_height_norm"] = targets[split]["actual_mean_height_norm"]
    table["baseline_mean_height_norm"] = targets[split]["baseline_mean_height_norm"]
    table["predicted_mean_height_norm"] = predicted_height
    table["absolute_error_norm"] = np.abs(
        table["actual_mean_height_norm"] - table["predicted_mean_height_norm"]
    )
    prediction_tables.append(table)

metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_tables, ignore_index=True)
four_context_path = REPO_ROOT / "outputs/masked_morphology/xgboost_height_only_v1/metrics.csv"
if four_context_path.exists():
    four_context = pd.read_csv(four_context_path)
    four_context["method"] = "xgboost_four_context"
    metrics = pd.concat([metrics, four_context], ignore_index=True, sort=False)

display(metrics[[
    "split", "method", "baseline_mae", "model_mae", "model_rmse",
    "model_within_5pct_reference_percent",
    "model_within_10pct_reference_percent", "model_r2",
]].round(4))

In [ ]:
test_metrics = metrics.loc[metrics["split"].eq("test")].set_index("method")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
test_metrics["model_mae"].plot.bar(ax=axes[0], color="#219ebc")
axes[0].set(title="Test normalized MAE", ylabel="Lower is better")
test_metrics[[
    "model_within_5pct_reference_percent",
    "model_within_10pct_reference_percent",
]].plot.bar(ax=axes[1], color=["#ffb703", "#219ebc"])
axes[1].set(title="Test tolerance accuracy", ylabel="Higher is better", ylim=(0, 100))
for ax in axes:
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 6. Inspect which vertebrae the model used

Distant offsets are often absent, so importance should be considered together with the percentage of training rows where an offset exists.

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X["train"].columns,
    "importance": model.feature_importances_,
    "observed_train_percent": 100 * X["train"].notna().mean().to_numpy(),
}).sort_values("importance", ascending=False)
display(feature_importance.head(25).round(4))

top = feature_importance.head(20).sort_values("importance")
ax = top.plot.barh(x="feature", y="importance", figsize=(9, 7), legend=False)
ax.set_title("Top all-vertebra XGBoost features")
plt.show()

## 7. Save and verify artifacts

In [ ]:
if SAVE_ARTIFACTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_model(OUTPUT_DIR / "mean_height_all_vertebrae.json")
    metrics.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
    with (OUTPUT_DIR / "training_history.json").open("w", encoding="utf-8") as file:
        json.dump(history, file, indent=2)
        file.write("\n")
    manifest = {
        "task": "all-visible-vertebrae masked mean height residual regression",
        "research_use_only": True,
        "feature_columns": list(X["train"].columns),
        "missing_value_contract": "NaN means no visible vertebra at that offset",
        "target_exclusion_contract": "relative offset 0 is never an input",
        "best_iteration": int(model.best_iteration),
        "model_parameters": MODEL_PARAMS,
    }
    with (OUTPUT_DIR / "run_manifest.json").open("w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)
        file.write("\n")

    reloaded = XGBRegressor()
    reloaded.load_model(OUTPUT_DIR / "mean_height_all_vertebrae.json")
    assert np.isfinite(reloaded.predict(X["test"].head(10))).all()
    print("Saved and reloaded:", OUTPUT_DIR)

## Decision rule

Keep the all-vertebra model only if its held-out gain over four-context XGBoost is meaningful. A tiny difference does not justify a much larger sparse input table.